## K-Nearest Neighbors (KNN)

O [KNN](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) é um algoritmo baseado em instâncias, que classifica uma nova observação com base nos seus vizinhos mais próximos.

A classe é definida pela maioria dos vizinhos.

### Hiperparâmetros utilizados

- **n_neighbors**: número de vizinhos considerados (default = 5)
  - Valores menores → modelo mais sensível 
  - Valores maiores → modelo mais estável

- **weights**: forma de ponderação
  - `uniform`: todos os vizinhos têm o mesmo peso (uniform)
  - `distance`: vizinhos mais próximos têm maior peso

### Estratégia

Foi utilizado GridSearchCV com validação cruzada para encontrar os melhores parâmetros, utilizando ROC AUC como métrica.

In [1]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

tuning_results = []

for scenario in scenarios:
    print(f"\n==========================================")
    print(f"🔎 Scenario: {scenario}")
    print(f"==========================================")
    
    import pandas as pd
    import sys
    import os

    # add raiz do projeto
    sys.path.append(os.path.abspath(".."))

    from sklearn.model_selection import GridSearchCV
    from sklearn.preprocessing import StandardScaler
    from imblearn.pipeline import Pipeline
    from imblearn.over_sampling import SMOTE
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

    from preprocessing.main_preprocessing import load_and_preprocess
    from utils.experiment_logger import save_results


save_results(tuning_results, file_path="../results/model_results.csv")

display(pd.DataFrame(tuning_results))



🔎 Scenario: sem_submodalidade

🔎 Scenario: submodalidade_agrupada

🔎 Scenario: submodalidade_engineered


""


## BASELINE

In [2]:
# scenarios = [
#     "sem_submodalidade",
#     "submodalidade_agrupada",
#     "submodalidade_engineered"
# ]

# smote_options = [False, True]

# results = []

# for scenario in scenarios:
#     for use_smote in smote_options:

#         print(f"\n🔎 Scenario: {scenario} | SMOTE: {use_smote}")

#         X_train, X_test, y_train, y_test = load_and_preprocess(
#             "../predictive_models/scrdata_202505.csv",
#             scenario=scenario,
#             use_smote=use_smote
#         )

#         steps = []
#         steps.append(('scaler', StandardScaler()))
#         if use_smote:
#             steps.append(('smote', SMOTE(random_state=42)))
#         steps.append(('knn', KNeighborsClassifier()))

#         model = Pipeline(steps)

#         model.fit(X_train, y_train)

#         y_pred = model.predict(X_test)
#         y_proba = model.predict_proba(X_test)[:, 1]

#         results.append({
#             "model": "KNN",
#             "scenario": scenario,
#             "smote": use_smote,
#             "roc_auc": roc_auc_score(y_test, y_proba),
#             "f1": f1_score(y_test, y_pred),
#             "accuracy": accuracy_score(y_test, y_pred),
#             "n_features": X_train.shape[1],
#             "phase": "baseline",
#             "timestamp": pd.Timestamp.now()
#         })

# save_results(results, file_path="../results/experiments.csv")

# df_result = pd.DataFrame(results)

# display(df_result)


## GRIDSEARCH

In [3]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

tuning_results = []

for scenario in scenarios:
    print(f"\n==========================================")
    print(f"🔎 Scenario: {scenario}")
    print(f"==========================================")
    
    X_train, X_test, y_train, y_test = load_and_preprocess(
        "../predictive_models/scrdata_202505.csv",
        scenario=scenario,
        use_smote=False
    )


    param_grid_knn = {
        "smote": [SMOTE(random_state=42), "passthrough"],
        "knn__n_neighbors": [3, 5, 7, 9],
        "knn__weights": ["uniform", "distance"],
        "knn__metric": ["euclidean", "manhattan"]
    }

    grid_knn = GridSearchCV(
        estimator=Pipeline([
            ('scaler', StandardScaler()),
            ('smote', SMOTE(random_state=42)),
            ('knn', KNeighborsClassifier())
        ]),
        param_grid=param_grid_knn,
        cv=5,
        scoring="roc_auc",
        n_jobs=-1
    )

    grid_knn.fit(X_train, y_train)

    print("Best parameters:", grid_knn.best_params_)
    print("Best ROC AUC (CV):", grid_knn.best_score_)


    #? BEST MODEL TEST EVALUATION

    best_knn = grid_knn.best_estimator_

    y_pred = best_knn.predict(X_test)
    y_proba = best_knn.predict_proba(X_test)[:, 1]


    #? TUNING (CV)



    cv_results = pd.DataFrame(grid_knn.cv_results_)
    cv_results['smote_used'] = cv_results['param_smote'].apply(lambda x: x != 'passthrough')

    for smote_val in [True, False]:
        sub_results = cv_results[cv_results['smote_used'] == smote_val]
        if not sub_results.empty:
            best_row = sub_results.sort_values('mean_test_score', ascending=False).iloc[0]
            params = best_row['params']

            tuning_results.append({
                "model": "KNN",
                "scenario": scenario,
                "smote": smote_val,
                "phase": "tuning_cv",
                "roc_auc": best_row['mean_test_score'],
                "f1": None,
                "accuracy": None,
                "best_params": str(params),
                "timestamp": pd.Timestamp.now()
            })

            # Re-fit the best model for this group to get test metrics
            best_model_group = grid_knn.estimator.set_params(**params)
            best_model_group.fit(X_train, y_train)

            y_pred = best_model_group.predict(X_test)
            y_proba = best_model_group.predict_proba(X_test)[:, 1]

            tuning_results.append({
                "model": "KNN",
                "scenario": scenario,
                "smote": smote_val,
                "phase": "test",
                "roc_auc": roc_auc_score(y_test, y_proba),
                "f1": f1_score(y_test, y_pred),
                "accuracy": accuracy_score(y_test, y_pred),
                "best_params": str(params),
                "timestamp": pd.Timestamp.now()
            })

    save_results(tuning_results, file_path="../results/model_results.csv")

    display(pd.DataFrame(tuning_results))

save_results(tuning_results, file_path="../results/model_results.csv")

display(pd.DataFrame(tuning_results))



🔎 Scenario: sem_submodalidade
Best parameters: {'knn__metric': 'manhattan', 'knn__n_neighbors': 9, 'knn__weights': 'distance', 'smote': 'passthrough'}
Best ROC AUC (CV): 0.8212607300877514


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,KNN,sem_submodalidade,True,tuning_cv,0.812124,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 06:57:20.278666
1,KNN,sem_submodalidade,True,test,0.814885,0.769677,0.737708,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 06:58:51.386397
2,KNN,sem_submodalidade,False,tuning_cv,0.821261,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 06:58:51.387273
3,KNN,sem_submodalidade,False,test,0.823874,0.791512,0.746979,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 07:00:04.347483



🔎 Scenario: submodalidade_agrupada
Best parameters: {'knn__metric': 'manhattan', 'knn__n_neighbors': 9, 'knn__weights': 'uniform', 'smote': 'passthrough'}
Best ROC AUC (CV): 0.8671545791091351


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,KNN,sem_submodalidade,True,tuning_cv,0.812124,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 06:57:20.278666
1,KNN,sem_submodalidade,True,test,0.814885,0.769677,0.737708,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 06:58:51.386397
2,KNN,sem_submodalidade,False,tuning_cv,0.821261,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 06:58:51.387273
3,KNN,sem_submodalidade,False,test,0.823874,0.791512,0.746979,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 07:00:04.347483
4,KNN,submodalidade_agrupada,True,tuning_cv,0.859195,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 07:34:48.197819
5,KNN,submodalidade_agrupada,True,test,0.859079,0.800400,0.774211,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 07:37:14.366454
6,KNN,submodalidade_agrupada,False,tuning_cv,0.867155,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 07:37:14.367820
7,KNN,submodalidade_agrupada,False,test,0.867868,0.823255,0.785928,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 07:39:10.837609



🔎 Scenario: submodalidade_engineered
Best parameters: {'knn__metric': 'manhattan', 'knn__n_neighbors': 9, 'knn__weights': 'distance', 'smote': 'passthrough'}
Best ROC AUC (CV): 0.8212607300877514


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,KNN,sem_submodalidade,True,tuning_cv,0.812124,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 06:57:20.278666
1,KNN,sem_submodalidade,True,test,0.814885,0.769677,0.737708,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 06:58:51.386397
2,KNN,sem_submodalidade,False,tuning_cv,0.821261,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 06:58:51.387273
3,KNN,sem_submodalidade,False,test,0.823874,0.791512,0.746979,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 07:00:04.347483
4,KNN,submodalidade_agrupada,True,tuning_cv,0.859195,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 07:34:48.197819
5,KNN,submodalidade_agrupada,True,test,0.859079,0.800400,0.774211,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 07:37:14.366454
6,KNN,submodalidade_agrupada,False,tuning_cv,0.867155,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 07:37:14.367820
7,KNN,submodalidade_agrupada,False,test,0.867868,0.823255,0.785928,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 07:39:10.837609
8,KNN,submodalidade_engineered,True,tuning_cv,0.812124,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 08:06:14.956059
9,KNN,submodalidade_engineered,True,test,0.814885,0.769677,0.737708,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 08:07:49.550537


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,KNN,sem_submodalidade,True,tuning_cv,0.812124,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 06:57:20.278666
1,KNN,sem_submodalidade,True,test,0.814885,0.769677,0.737708,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 06:58:51.386397
2,KNN,sem_submodalidade,False,tuning_cv,0.821261,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 06:58:51.387273
3,KNN,sem_submodalidade,False,test,0.823874,0.791512,0.746979,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 07:00:04.347483
4,KNN,submodalidade_agrupada,True,tuning_cv,0.859195,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 07:34:48.197819
5,KNN,submodalidade_agrupada,True,test,0.859079,0.800400,0.774211,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 07:37:14.366454
6,KNN,submodalidade_agrupada,False,tuning_cv,0.867155,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 07:37:14.367820
7,KNN,submodalidade_agrupada,False,test,0.867868,0.823255,0.785928,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 07:39:10.837609
8,KNN,submodalidade_engineered,True,tuning_cv,0.812124,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 08:06:14.956059
9,KNN,submodalidade_engineered,True,test,0.814885,0.769677,0.737708,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-05-20 08:07:49.550537
